# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [22]:
# unfortunatly the last experiment runs out of time on the free T4 credit I have
# I therefore use my computer's 5070 laptop for this and change the corresponding lines
import gdown
gdown.download(id="17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes", output="linf_base.pt", quiet=False)
gdown.download(id="1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe", output="cifar10_train_10k.pt", quiet=False)
gdown.download(id="1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA", output="cifar10_test_1k.pt", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: C:\Users\yichaoy2\linf_base.pt
100%|██████████| 11.2M/11.2M [00:01<00:00, 10.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=1d701b50-ce38-41a7-ace2-af5be4a56488
To: C:\Users\yichaoy2\cifar10_train_10k.pt
100%|██████████| 30.8M/30.8M [00:03<00:00, 9.37MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: C:\Users\yichaoy2\cifar10_test_1k.pt
100%|██████████| 3.08M/3.08M [00:00<00:00, 9.05MB/s]


'cifar10_test_1k.pt'

In [6]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [7]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [8]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [9]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    # copied from Q1
    delta = torch.zeros_like(x)

    if rand_init:
        delta.uniform_(-eps, eps)
    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-10) * eps * r
    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        direction = grad / (l2_norm(grad) + 1e-10)
        delta = delta + alpha * direction
        scale = (eps / l2_norm(delta).clamp_min(1e-10)).clamp(max=1.0)
        delta = delta * scale
        delta = clip_box(x, delta)

    return (x + delta).detach()


In [10]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [11]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [12]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.5  l2 59.9  l1 23.3  union 23.2   (33 s)


{'clean': 78.7, 'linf': 48.5, 'l2': 59.9, 'l1': 23.3, 'union': 23.2}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [13]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [14]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    if norm == "linf":
        model.eval()
        x_adv = attack(model, x, y, norm)
        model.train()
        return F.cross_entropy(model(x_adv), y)
    elif norm == "l2":
        model.eval()
        x_adv = attack(model, x, y, norm)
        model.train()
        return F.cross_entropy(model(x_adv), y)
    elif norm == "l1":
        model.eval()
        x_adv = attack(model, x, y, norm)
        model.train()
        return F.cross_entropy(model(x_adv), y)
    else:
        raise NotImplementedError

In [15]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.199 (14 s)
epoch 2: loss 1.178 (29 s)
epoch 3: loss 1.153 (44 s)
epoch 4: loss 1.120 (58 s)
epoch 5: loss 1.102 (73 s)
epoch 6: loss 1.071 (88 s)
epoch 7: loss 1.044 (103 s)
epoch 8: loss 1.025 (117 s)
epoch 9: loss 1.011 (131 s)
epoch 10: loss 1.004 (145 s)
clean 77.7  linf 44.5  l2 57.4  l1 20.4  union 20.3   (32 s)


{'clean': 77.7, 'linf': 44.5, 'l2': 57.4, 'l1': 20.4, 'union': 20.3}

In [16]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.898 (14 s)
epoch 2: loss 0.833 (27 s)
epoch 3: loss 0.791 (42 s)
epoch 4: loss 0.748 (56 s)
epoch 5: loss 0.710 (70 s)
epoch 6: loss 0.670 (85 s)
epoch 7: loss 0.640 (100 s)
epoch 8: loss 0.612 (114 s)
epoch 9: loss 0.589 (129 s)
epoch 10: loss 0.578 (143 s)
clean 82.7  linf 32.4  l2 60.1  l1 29.0  union 25.3   (31 s)


{'clean': 82.7, 'linf': 32.4, 'l2': 60.1, 'l1': 29.0, 'union': 25.3}

In [17]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.229 (16 s)
epoch 2: loss 0.892 (31 s)
epoch 3: loss 0.616 (46 s)
epoch 4: loss 0.455 (61 s)
epoch 5: loss 0.314 (76 s)
epoch 6: loss 0.256 (91 s)
epoch 7: loss 0.214 (106 s)
epoch 8: loss 0.176 (121 s)
epoch 9: loss 0.160 (136 s)
epoch 10: loss 0.152 (151 s)
clean 84.2  linf 0.0  l2 0.2  l1 27.3  union 0.0   (31 s)


{'clean': 84.2, 'linf': 0.0, 'l2': 0.2, 'l1': 27.3, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [18]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    z_q, z_r = model(x_q), model(x_r)
    loss_q = F.cross_entropy(z_q, y, reduction="none")
    loss_r = F.cross_entropy(z_r, y, reduction="none")
    mask = z_q.argmax(1) == y
    kl = 0.0
    if mask.any():
        kl = F.kl_div(F.log_softmax(z_r[mask], dim=1),
                      F.softmax(z_q[mask], dim=1), reduction="batchmean")
    return torch.maximum(loss_q, loss_r).mean() + lam * kl


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    cos = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0, eps=1e-10)
    return cos.clamp_min(0) * g_n

In [19]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.432 (33 s)
epoch 2: loss 1.337 (65 s)
epoch 3: loss 1.300 (97 s)
epoch 4: loss 1.273 (130 s)
epoch 5: loss 1.249 (162 s)
epoch 6: loss 1.219 (194 s)
epoch 7: loss 1.205 (226 s)
epoch 8: loss 1.186 (258 s)
epoch 9: loss 1.172 (290 s)
epoch 10: loss 1.168 (321 s)
clean 78.6  linf 41.8  l2 62.7  l1 45.0  union 39.9   (31 s)


{'clean': 78.6, 'linf': 41.8, 'l2': 62.7, 'l1': 45.0, 'union': 39.9}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

## Proposed solution MAX + multi steepest descent

1. for each sample, we generate a new adversarail sample based on https://proceedings.mlr.press/v119/maini20a/maini20a.pdf's algorithm 1: Multi steepest descent. At every step of delta, we update based on the norm that causes the largest loss from l1, l2 and linf

2. further, we pick the adversirial sample based on either the new sample and linf PGD. This is becuase in ramp, the linf has the lowest robust accuracy (which combines all three norms) so we hope that linf PGD adversarial samples can help retain the robustness.


In [20]:
def linf_step(x, delta, grad, eps, alpha):
    # Copied from the loop in pgd_linf.
    delta = delta + alpha * grad.sign()
    delta = delta.clamp(-eps, eps)
    delta = clip_box(x, delta)
    return delta


def l2_step(x, delta, grad, eps, alpha):
    # Copied from the loop in pgd_l2.
    direction = grad / (l2_norm(grad) + 1e-10)
    delta = delta + alpha * direction
    scale = (eps / l2_norm(delta).clamp_min(1e-10)).clamp(max=1.0)
    delta = delta * scale
    delta = clip_box(x, delta)
    return delta


def l1_step(x, delta, grad, eps, alpha, k=20):
    # Copied from the loop in the starter's pgd_l1.
    grad = grad.flatten(1)
    x_cur = (x + delta).flatten(1)
    blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
    grad = grad.masked_fill(blocked, 0)
    top = grad.abs().topk(k, dim=1).indices
    step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
    delta = clip_box(x, delta + alpha * (step / k).view_as(x))
    delta = project_l1(delta, eps)
    return delta


def msd_attack(model, x, y, steps=10, k=20):
    # Maini et al., Algorithm 1: https://proceedings.mlr.press/v119/maini20a/maini20a.pdf
    delta = torch.zeros_like(x)
    rows = torch.arange(len(x), device=x.device)
    # Same step sizes as the starter's attack().
    alpha_linf = 2.5 * EPS["linf"] / steps
    alpha_l2 = 2.5 * EPS["l2"] / steps
    alpha_l1 = EPS["l1"] / steps ** 0.5

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        with torch.no_grad():
            candidates = torch.stack([
                linf_step(x, delta, grad, EPS["linf"], alpha_linf),
                l2_step(x, delta, grad, EPS["l2"], alpha_l2),
                l1_step(x, delta, grad, EPS["l1"], alpha_l1, k),
            ])
            losses = torch.stack([
                F.cross_entropy(model(x + d), y, reduction="none")
                for d in candidates
            ])
            delta = candidates[losses.argmax(0), rows]
    return (x + delta).detach()


def my_loss(model, x, y):
    # TODO: your method
    # MAX over MSD and L-infinity PGD; same structure as ramp_loss without KL.
    model.eval()
    x_msd = msd_attack(model, x, y)
    x_linf = attack(model, x, y, "linf")
    model.train()
    loss_msd = F.cross_entropy(model(x_msd), y, reduction="none")
    loss_linf = F.cross_entropy(model(x_linf), y, reduction="none")
    return torch.maximum(loss_msd, loss_linf).mean()

In [27]:
# seed 0
seed = 0

# mine
model_mine_0 = adv_train(load_base_model(), my_loss, seed=seed)
test_model(model_mine_0)

# RAMP
model_ramp_0 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=seed)
test_model(model_ramp_0)

epoch 1: loss 1.398 (38 s)
epoch 2: loss 1.304 (76 s)
epoch 3: loss 1.268 (115 s)
epoch 4: loss 1.233 (154 s)
epoch 5: loss 1.214 (193 s)
epoch 6: loss 1.182 (232 s)
epoch 7: loss 1.155 (272 s)
epoch 8: loss 1.137 (311 s)
epoch 9: loss 1.125 (349 s)
epoch 10: loss 1.118 (388 s)
clean 76.8  linf 44.1  l2 61.5  l1 46.5  union 41.8   (31 s)
epoch 1: loss 1.431 (33 s)
epoch 2: loss 1.337 (65 s)
epoch 3: loss 1.301 (98 s)
epoch 4: loss 1.273 (130 s)
epoch 5: loss 1.249 (161 s)
epoch 6: loss 1.219 (194 s)
epoch 7: loss 1.205 (226 s)
epoch 8: loss 1.186 (258 s)
epoch 9: loss 1.172 (291 s)
epoch 10: loss 1.168 (323 s)
clean 78.3  linf 41.7  l2 62.8  l1 44.8  union 39.5   (31 s)


{'clean': 78.3, 'linf': 41.7, 'l2': 62.8, 'l1': 44.8, 'union': 39.5}

In [28]:
# seed 1
seed = 1

# mine
model_mine_1 = adv_train(load_base_model(), my_loss, seed=seed)
test_model(model_mine_1)

# RAMP
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=seed)
test_model(model_ramp_1)

epoch 1: loss 1.405 (38 s)
epoch 2: loss 1.300 (77 s)
epoch 3: loss 1.266 (116 s)
epoch 4: loss 1.239 (154 s)
epoch 5: loss 1.214 (193 s)
epoch 6: loss 1.183 (231 s)
epoch 7: loss 1.161 (270 s)
epoch 8: loss 1.143 (308 s)
epoch 9: loss 1.129 (346 s)
epoch 10: loss 1.121 (386 s)
clean 76.7  linf 43.5  l2 61.9  l1 47.1  union 41.5   (31 s)
epoch 1: loss 1.432 (33 s)
epoch 2: loss 1.328 (65 s)
epoch 3: loss 1.299 (98 s)
epoch 4: loss 1.272 (129 s)
epoch 5: loss 1.248 (161 s)
epoch 6: loss 1.228 (192 s)
epoch 7: loss 1.204 (224 s)
epoch 8: loss 1.184 (256 s)
epoch 9: loss 1.169 (288 s)
epoch 10: loss 1.172 (321 s)
clean 78.2  linf 42.0  l2 62.4  l1 45.3  union 40.0   (31 s)


{'clean': 78.2, 'linf': 42.0, 'l2': 62.4, 'l1': 45.3, 'union': 40.0}

## Ablation Study

Here the ablation study evaluate two other designs
1. only msd
2. only MAX with l1 l2 linf

the result under seed 0 and 1 are reported

In [30]:
# msd only
def msd_only_loss(model, x, y):
    model.eval()
    x_adv = msd_attack(model, x, y)
    model.train()
    return F.cross_entropy(model(x_adv), y)

# seed 0
model_msd_0 = adv_train(load_base_model(), msd_only_loss, seed=0)
test_model(model_msd_0)

# seed 1
model_msd_1 = adv_train(load_base_model(), msd_only_loss, seed=1)
test_model(model_msd_1)

epoch 1: loss 1.382 (25 s)
epoch 2: loss 1.276 (51 s)
epoch 3: loss 1.241 (78 s)
epoch 4: loss 1.213 (104 s)
epoch 5: loss 1.182 (130 s)
epoch 6: loss 1.155 (157 s)
epoch 7: loss 1.130 (183 s)
epoch 8: loss 1.109 (209 s)
epoch 9: loss 1.095 (235 s)
epoch 10: loss 1.095 (261 s)
clean 77.3  linf 42.4  l2 61.6  l1 49.4  union 41.5   (31 s)
epoch 1: loss 1.379 (26 s)
epoch 2: loss 1.274 (52 s)
epoch 3: loss 1.241 (78 s)
epoch 4: loss 1.213 (104 s)
epoch 5: loss 1.180 (131 s)
epoch 6: loss 1.155 (157 s)
epoch 7: loss 1.127 (183 s)
epoch 8: loss 1.112 (210 s)
epoch 9: loss 1.101 (236 s)
epoch 10: loss 1.088 (263 s)
clean 77.3  linf 42.5  l2 62.6  l1 48.5  union 41.8   (31 s)


{'clean': 77.3, 'linf': 42.5, 'l2': 62.6, 'l1': 48.5, 'union': 41.8}

In [31]:
# MAX only (pick the largest loss among l1 l2 linf)
def max_only_loss(model, x, y):
    model.eval()
    x_adv = [attack(model, x, y, norm) for norm in NORMS]
    model.train()
    losses = torch.stack([F.cross_entropy(model(z), y, reduction="none") for z in x_adv])
    return losses.max(dim=0).values.mean()

# seed 0
model_max_0 = adv_train(load_base_model(), max_only_loss, seed=0)
test_model(model_max_0)

# seed 1
model_max_1 = adv_train(load_base_model(), max_only_loss, seed=1)
test_model(model_max_1)

epoch 1: loss 1.404 (44 s)
epoch 2: loss 1.306 (89 s)
epoch 3: loss 1.264 (133 s)
epoch 4: loss 1.240 (177 s)
epoch 5: loss 1.207 (221 s)
epoch 6: loss 1.183 (266 s)
epoch 7: loss 1.157 (310 s)
epoch 8: loss 1.136 (354 s)
epoch 9: loss 1.120 (398 s)
epoch 10: loss 1.121 (442 s)
clean 76.3  linf 43.3  l2 61.5  l1 46.1  union 41.0   (31 s)
epoch 1: loss 1.406 (44 s)
epoch 2: loss 1.302 (89 s)
epoch 3: loss 1.261 (133 s)
epoch 4: loss 1.238 (177 s)
epoch 5: loss 1.204 (222 s)
epoch 6: loss 1.180 (266 s)
epoch 7: loss 1.158 (310 s)
epoch 8: loss 1.138 (354 s)
epoch 9: loss 1.119 (399 s)
epoch 10: loss 1.116 (443 s)
clean 76.8  linf 43.5  l2 62.1  l1 47.1  union 41.5   (31 s)


{'clean': 76.8, 'linf': 43.5, 'l2': 62.1, 'l1': 47.1, 'union': 41.5}